# 01. Engineering Data Reality Check
**Engineering Data Processing with Python** | one-day course | approx. 35 minutes

> **SOLUTION NOTEBOOK.** Every blank and TODO is filled in. Try the exercise version first; use this when you are stuck or to compare approaches. There is usually more than one correct answer.


In [1]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = ""   # if your trainer gives you a download link for data.zip, paste it between the quotes

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## Why this module exists

Most engineering data problems are boring and predictable. Someone typed `???` in a cell. An export changed the date
format. A sensor wrote `-999` when it lost signal. A column name has a trailing space you cannot see.
None of these are hard to fix once you have seen them. The trouble is not seeing them, then reporting a number
that is confidently wrong.

In this module you will:

1. Look at the failure modes you will meet all day.
2. Agree some rules that make your work repeatable.
3. Profile two real datasets and find what is wrong with them.
4. Write a reusable `profile()` function.
5. Decide what "clean enough" means for a specific question.

## 1. Common failure modes in engineering exports

| Failure mode | What it looks like | Why it bites |
|---|---|---|
| Numbers stored as text | `"15 kW"`, `"€1,250.00"`, `"???"` | Column will not average or sum |
| Date order | `01/02/2026`: 1 Feb or 2 Jan? | Silent. Monthly reports shift |
| Mixed units | °F from one logger, °C from another | Silent. Plausible but wrong |
| Sentinel values | `-999`, `9999`, `0` for "no reading" | Drags averages, triggers false alarms |
| Hidden whitespace and case | `"CNC-001 "`, `"cnc-001"` | Joins and counts miss records |
| Header junk | Title rows above the real header in Excel | Column names become `Unnamed: 1` |
| Duplicates | Same record exported twice, or edited and re-exported | Counts and costs inflated |
| Free-text categories | `"Corrective"`, `"CM"`, `"breakdown"` | Group totals split three ways |
| Excel "helping" | `1-3` becomes a date, leading zeros dropped | IDs no longer match |

## 2. Rules for repeatable work

1. **Raw files are read-only.** Never fix the source file by hand. Fix it in code so the fix is recorded and repeatable.
2. **Every step is in the notebook.** If you clicked something in Excel to make it work, you cannot rerun it next month.
3. **Run top to bottom.** Before you trust a result: *Runtime > Restart session and run all*. If it breaks, your
   notebook depended on something you ran out of order.
4. **Outputs go to a separate folder** (`outputs/`), never over the input.
5. **Count before and after** every cleaning step. Rows that disappear without anyone noticing are the commonest bug.

## 3. Profiling: first look at a dataset

The 3D printing dataset from the primer. Remember that `roughness` would not average.

### Exercise 1.1 [Level 1: fill the blanks]
Load the file and look at its size, its first rows and the type of each column.

In [2]:
printer = pd.read_csv("data/3dprinter.csv")
print("Shape (rows, columns):", printer.shape)
printer.head()

Shape (rows, columns): (51, 12)


,layer_height,wall_thickness,infill_density,infill_pattern,nozzle_temperature,bed_temperature,print_speed,material,fan_speed,roughness,tension_strenght,elongation
0,0.02,8,90,grid,220,60,40,abs,0,25,18.0,1.2
1,0.02,7,90,honeycomb,225,65,40,abs,25,32,16.0,1.4
2,0.02,1,80,grid,230,70,40,abs,50,40,8.0,0.8
3,0.02,4,70,honeycomb,240,75,40,abs,75,68,10.0,0.5
4,0.02,6,90,grid,250,80,40,abs,100,92,5.0,0.7


In [3]:
printer.info()   # column names, non-null counts and dtypes in one view

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   layer_height        51 non-null     float64
 1   wall_thickness      51 non-null     int64  
 2   infill_density      51 non-null     int64  
 3   infill_pattern      51 non-null     object 
 4   nozzle_temperature  51 non-null     int64  
 5   bed_temperature     51 non-null     int64  
 6   print_speed         51 non-null     int64  
 7   material            51 non-null     object 
 8   fan_speed           51 non-null     int64  
 9   roughness           51 non-null     object 
 10  tension_strenght    50 non-null     float64
 11  elongation          50 non-null     float64
dtypes: float64(3), int64(6), object(3)
memory usage: 4.9+ KB


`roughness` shows as `object`. In pandas, `object` usually means **text**. A numeric column becomes text if even one
value cannot be read as a number. Let's find the culprit.

In [4]:
# to_numeric with errors="coerce" turns anything that is not a number into NaN (missing).
roughness_num = pd.to_numeric(printer["roughness"], errors="coerce")

# Rows where the original had a value but the number version is missing = the bad values
bad_rows = printer[roughness_num.isna() & printer["roughness"].notna()]
bad_rows

,layer_height,wall_thickness,infill_density,infill_pattern,nozzle_temperature,bed_temperature,print_speed,material,fan_speed,roughness,tension_strenght,elongation
50,0.1,5,40,grid,210,70,50,pla,120,???,NaN,NaN


One test row has `???` for roughness and blanks for tensile strength and elongation. Someone was clearly unsure.
Two more things to notice:

* `tension_strenght` is misspelt. If you write `printer["tension_strength"]` you get a `KeyError`.
  Misspelt column names in exports are common and they break scripts in confusing ways.
* The bad row is the last row. Exports often have junk at the end: totals, notes, partial records.

### Exercise 1.2 [Level 1: fill the blanks]
Count missing values per column, fix the column name, and convert roughness to numbers.

In [5]:
print(printer.isna().sum())

printer = printer.rename(columns={"tension_strenght": "tension_strength"})
printer["roughness"] = pd.to_numeric(printer["roughness"], errors="coerce")
printer.dtypes

layer_height          0
wall_thickness        0
infill_density        0
infill_pattern        0
nozzle_temperature    0
bed_temperature       0
print_speed           0
material              0
fan_speed             0
roughness             0
tension_strenght      1
elongation            1
dtype: int64


layer_height          float64
wall_thickness          int64
infill_density          int64
infill_pattern         object
nozzle_temperature      int64
bed_temperature         int64
print_speed             int64
material               object
fan_speed               int64
roughness             float64
tension_strength      float64
elongation            float64
dtype: object

In [6]:
check("tension_strength" in printer.columns and printer["roughness"].dtype == "float64",
      "Column renamed and roughness is numeric. The ??? is now NaN, which pandas ignores in averages.",
      "Check the new column name spelling and errors='coerce'.")

PASS: Column renamed and roughness is numeric. The ??? is now NaN, which pandas ignores in averages.


In [7]:
# Now this works
printer.groupby("material")["roughness"].mean()

material
abs    193.44
pla    147.72
Name: roughness, dtype: float64

**Decision point.** We turned `???` into missing. We did not delete the row, and we did not guess a value.
That is usually the safest default: keep the evidence, make the gap visible, decide later what to do with it.

## 4. A bigger dataset: concrete compressive strength

### Exercise 1.3 [Level 1: fill the blanks]

In [8]:
concrete = pd.read_csv("data/concrete.csv")
print(concrete.shape)
concrete.describe().T     # .T flips the table so each column is a row. Easier to read.

(1030, 9)


,count,mean,std,min,25%,50%,75%,max
Cement (component 1)(kg in a m^3 mixture),1030.0,281.165631,104.507142,102.000000,192.375000,272.900000,350.000000,540.000000
Blast Furnace Slag (component 2)(kg in a m^3 mixture),1030.0,73.895485,86.279104,0.000000,0.000000,22.000000,142.950000,359.400000
Fly Ash (component 3)(kg in a m^3 mixture),1030.0,54.187136,63.996469,0.000000,0.000000,0.000000,118.270000,200.100000
Water (component 4)(kg in a m^3 mixture),1030.0,181.566359,21.355567,121.750000,164.900000,185.000000,192.000000,247.000000
Superplasticizer (component 5)(kg in a m^3 mixture),1030.0,6.203112,5.973492,0.000000,0.000000,6.350000,10.160000,32.200000
Coarse Aggregate (component 6)(kg in a m^3 mixture),1030.0,972.918592,77.753818,801.000000,932.000000,968.000000,1029.400000,1145.000000
Fine Aggregate (component 7)(kg in a m^3 mixture),1030.0,773.578883,80.175427,594.000000,730.950000,779.510000,824.000000,992.600000
Age (day),1030.0,45.662136,63.169912,1.000000,7.000000,28.000000,56.000000,365.000000
"Concrete compressive strength(MPa, megapascals)",1030.0,35.817836,16.705679,2.331808,23.707115,34.442774,46.136287,82.599225


The numbers look sensible (cement 102 to 540 kg/m³, strength 2 to 83 MPa). The column names are painful.
Print them with `repr` so hidden spaces show up.

In [9]:
for name in concrete.columns:
    print(repr(name))

'Cement (component 1)(kg in a m^3 mixture)'
'Blast Furnace Slag (component 2)(kg in a m^3 mixture)'
'Fly Ash (component 3)(kg in a m^3 mixture)'
'Water  (component 4)(kg in a m^3 mixture)'
'Superplasticizer (component 5)(kg in a m^3 mixture)'
'Coarse Aggregate  (component 6)(kg in a m^3 mixture)'
'Fine Aggregate (component 7)(kg in a m^3 mixture)'
'Age (day)'
'Concrete compressive strength(MPa, megapascals) '


Look closely: `'Water  (component 4)...'` has **two** spaces and the strength column ends with a **space** before the
closing quote. Type the name as you see it in Excel and it will not match.

### Exercise 1.4 [Level 1: fill the blanks]
Replace the column names with short, consistent names that include units. Two are left for you.

In [10]:
new_names = [
    "cement_kg_m3",
    "slag_kg_m3",
    "fly_ash_kg_m3",
    "water_kg_m3",
    "superplasticiser_kg_m3",
    "coarse_agg_kg_m3",
    "fine_agg_kg_m3",
    "age_days",
    "strength_mpa",
]
print(len(new_names), "new names for", len(concrete.columns), "columns")
concrete.columns = new_names
concrete.head()

9 new names for 9 columns


,cement_kg_m3,slag_kg_m3,fly_ash_kg_m3,water_kg_m3,superplasticiser_kg_m3,coarse_agg_kg_m3,fine_agg_kg_m3,age_days,strength_mpa
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


In [11]:
check(list(concrete.columns)[3] == "water_kg_m3" and list(concrete.columns)[-1] == "strength_mpa",
      "Readable names. Water/cement ratio is now easy to calculate.",
      "Expected 'water_kg_m3' and 'strength_mpa'.")

PASS: Readable names. Water/cement ratio is now easy to calculate.


Assigning a list of names relies on the **order** of columns. It is fine here because we checked the order by eye.
In a script that runs on new exports every month, renaming with a dictionary (`old name: new name`) is safer,
because if a column moves, the dictionary still finds it. You will do that in Module 03.

### Exercise 1.5 [Level 1: fill the blanks]
Two quick engineering checks: are there duplicate rows, and what is the water/cement ratio range?

In [12]:
print("Duplicate rows:", concrete.duplicated().sum())

concrete["w_c_ratio"] = concrete["water_kg_m3"] / concrete["cement_kg_m3"]
concrete["w_c_ratio"].describe()

Duplicate rows: 25


count    1030.000000
mean        0.748269
std         0.314005
min         0.266893
25%         0.533333
50%         0.675349
75%         0.935014
max         1.882353
Name: w_c_ratio, dtype: float64

25 duplicate rows. Are they errors, or genuine repeat tests of the same mix at the same age? You cannot tell from the
data alone. Keep that question for Module 04.

## 5. A reusable profiling function

You will profile every new dataset. Write it once.

### Exercise 1.6 [Level 2: finish the code]

Complete `profile(df)` so it returns one row per column of `df` with:
`dtype`, `missing` (count), `missing_pct`, `unique` (number of distinct values) and `example` (first non-missing value).

Useful pieces: `df.dtypes`, `df.isna().sum()`, `df.isna().mean()`, `df.nunique()`. The `example` line is written for you.

In [13]:
def profile(df):
    """One row per column: type, missing count and %, distinct values, an example value."""
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(),
        "example": df.apply(lambda col: col.dropna().iloc[0] if col.notna().any() else None),
    })
    return summary

profile(printer)

,dtype,missing,missing_pct,unique,example
layer_height,float64,0,0.0,5,0.02
wall_thickness,int64,0,0.0,10,8
infill_density,int64,0,0.0,9,90
infill_pattern,object,0,0.0,2,grid
nozzle_temperature,int64,0,0.0,9,220
bed_temperature,int64,0,0.0,5,60
print_speed,int64,0,0.0,4,40
material,object,0,0.0,2,abs
fan_speed,int64,0,0.0,7,0
roughness,float64,1,2.0,44,25.0


In [14]:
p = profile(printer)
check(set(["dtype", "missing", "missing_pct", "unique", "example"]).issubset(p.columns) and p.loc["roughness", "missing"] == 1,
      "profile() works. Keep it, you will use it again.",
      "Need columns dtype, missing, missing_pct, unique, example. roughness should show 1 missing.")

PASS: profile() works. Keep it, you will use it again.


## 6. A preview of the main dataset for today

For the rest of the day we work with data from **Kilbrack Engineering Ltd**, a fictional manufacturer with plants in
Cork, Galway and Limerick. The data is synthetic, but the faults planted in it are the common ones in real maintenance and asset data.
You will meet:

* `work_orders_cmms.csv`: maintenance work orders from the current maintenance system (CMMS)
* `work_orders_legacy.xlsx`: older work orders from the spreadsheet it replaced
* `asset_register.xlsx`: the list of assets
* `sensor_readings.csv`: condition monitoring data from compressors and pumps

Run your profiler on the work orders.

In [15]:
wo = pd.read_csv("data/work_orders_cmms.csv")
print(wo.shape)
profile(wo)

(1016, 12)


,dtype,missing,missing_pct,unique,example
wo_id,object,0,0.0,973,WO-25-00270
asset_id,object,0,0.0,159,PMP-005
wo_type,object,0,0.0,12,PM
priority,object,21,2.1,7,3
date_raised,object,0,0.0,967,23/06/2025 13:15
date_closed,object,41,4.0,955,24/06/2025 01:17
status,object,0,0.0,5,Closed
labour_hours,float64,36,3.5,32,2.5
parts_cost_eur,object,68,6.7,802,222.81
technician,object,0,0.0,32,Gráinne FitzGerald


In [16]:
# How many different spellings of work order type?
wo["wo_type"].value_counts()

wo_type
PM             321
CM             139
PM              96
Preventive      91
pm              88
Corrective      54
cm              50
INSP            47
corrective      44
Breakdown       42
insp            24
Inspection      20
Name: count, dtype: int64

Every column is `object` (text), including hours, cost and dates. There are 12 spellings of 3 work order types.
And `unique` for `wo_id` is lower than the number of rows, so some work orders appear more than once.
That is the next three modules sorted.

## 7. What is "clean enough"?

There is no such thing as perfectly clean data. There is data that is clean enough **for a specific question**.
Cleaning everything to perfection wastes days. Cleaning nothing produces nonsense.

### Exercise 1.7 [Level 3: think it through, then compare with Gemini]

The operations manager asks: **"Which asset type costs us the most in corrective maintenance?"**

In the markdown cell below (double-click to edit), list:
1. Which columns must be right to answer this?
2. Which problems in those columns would change the answer?
3. Which columns can you ignore for this question?

Then ask Gemini the same thing. Paste in the output of `profile(wo)` and `wo.head()`, not the whole file.
Compare. What did it spot that you missed? Did it assume anything that is not true (for example a column
that does not exist)?

*Model answer:*

1. **Must be right:** `wo_type` (to select corrective work), `parts_cost_eur` (and arguably `labour_hours` x a labour
   rate), `asset_id` (to link to the asset register), and the register's `asset_type`.
2. **Problems that change the answer:** type spellings (`CM`, `cm`, `Corrective`, `Breakdown` must all count as
   corrective); cost stored as text with `€` and commas; `N/A`, `-` and blanks in cost; duplicate work orders
   inflating totals; asset IDs with spaces or lower case failing to match the register; work orders on assets
   that are not in the register.
3. **Can ignore (for this question):** technician names, description text, priority, exact timestamps
   (unless the question is for a specific period, in which case dates matter too).
4. **Typical AI misses:** it may suggest filling missing costs with the average (which invents money), or assume
   one spelling per category. Ask it what it assumed.

## Summary

* `object` dtype on a numeric column means at least one value is text. `pd.to_numeric(..., errors="coerce")` finds it.
* Print column names with `repr()` to see hidden spaces.
* Duplicates are a judgement call, not an automatic delete.
* Profile every new dataset with the same function.
* Define "clean enough" against the question you need to answer.

Next: **02 Loading Engineering Datasets**.